# Ultrasound Segmentation — U-Net Training
ResNet-18 encoder (SWSL weights, layer4 unfrozen) · Dice loss · Metrics: Dice, Pixel Accuracy

## 1. Imports & Device

In [ ]:
import torch
import torch.optim as optim
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from data_loader import load_ultrasound_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Metrics

In [ ]:
def calculate_dice(preds, targets, smooth=1e-6):
    preds   = preds.view(-1)
    targets = targets.view(-1)
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)


def calculate_pixel_accuracy(preds, targets):
    return (preds == targets).sum().item() / targets.numel()

## 3. Model
ResNet-18 with SWSL pretrained weights. Encoder frozen except `layer4`.

In [ ]:
def get_model():
    model = smp.Unet(
        encoder_name="resnet34", 
        encoder_weights="imagenet", 
        in_channels=3, 
        classes=1
    ).to(device)

    return model

## 4. Training

In [ ]:
history = {'loss': [], 'dice': [], 'pixel_acc': []}


def train_model(model, loader, epochs=30):
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
    )
    criterion = smp.losses.DiceLoss(mode='binary')

    print(f'--- Training on {device} ---')
    model.train()

    for epoch in range(epochs):
        running_loss = running_dice = running_pa = 0.0

        for batch in loader:
            inputs = batch['image'].to(device)
            labels = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            running_loss += loss.item()
            running_dice += calculate_dice(preds, labels).item()
            running_pa   += calculate_pixel_accuracy(preds, labels)

        n = len(loader)
        epoch_loss = running_loss / n
        epoch_dice = running_dice / n
        epoch_pa   = running_pa   / n

        history['loss'].append(epoch_loss)
        history['dice'].append(epoch_dice)
        history['pixel_acc'].append(epoch_pa)

        print(
            f'Epoch {epoch+1:>3}/{epochs} '
            f'| Loss: {epoch_loss:.4f} '
            f'| Dice: {epoch_dice:.4f} '
            f'| PA: {epoch_pa:.4f}'
        )

## 5. Training Curves
Each metric in its own figure.

In [ ]:
def plot_training_curves(history):
    epochs = range(1, len(history['loss']) + 1)
    configs = [
        ('loss',      'Loss',           'Loss'),
        ('dice',      'Dice Score',     'Dice'),
        ('pixel_acc', 'Pixel Accuracy', 'Pixel Accuracy'),
    ]

    for key, title, ylabel in configs:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(epochs, history[key], color='steelblue', linewidth=1.8)
        ax.set_title(f'{title} over {len(epochs)} Epochs')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

## 6. Evaluation
Each test sample gets its own figure (image / ground truth / prediction).

In [ ]:
def evaluate_and_visualize(model, loader):
    model.eval()

    all_imgs, all_masks, all_preds = [], [], []
    all_dices, all_pixel_accs = [], []

    print('\n' + '=' * 50)
    print(f'{"Sample #":<12} | {"Dice":<10} | {"Pixel Acc"}')
    print('-' * 40)

    with torch.no_grad():
        count = 0
        for batch in loader:
            images = batch['image'].to(device)
            masks  = batch['mask'].to(device)
            preds  = (torch.sigmoid(model(images)) > 0.5).float()

            for i in range(images.size(0)):
                count += 1
                dice = calculate_dice(preds[i], masks[i]).item()
                pa   = calculate_pixel_accuracy(preds[i], masks[i])

                print(f'Image {count:02d}      | {dice:<10.4f} | {pa:.4f}')

                all_imgs.append(images[i].cpu())
                all_masks.append(masks[i].cpu())
                all_preds.append(preds[i].cpu())
                all_dices.append(dice)
                all_pixel_accs.append(pa)

    mean_dice = sum(all_dices) / len(all_dices)
    mean_pa   = sum(all_pixel_accs) / len(all_pixel_accs)
    print('-' * 40)
    print(f'MEAN         | {mean_dice:<10.4f} | {mean_pa:.4f}\n')

    # One figure per sample
    for i in range(len(all_imgs)):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(
            f'Sample {i+1}  |  Dice: {all_dices[i]:.4f}  |  Pixel Acc: {all_pixel_accs[i]:.4f}',
            fontsize=13,
        )

        img = all_imgs[i].permute(1, 2, 0).numpy()          # ← all_imgs, already on cpu
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        axes[0].imshow(img)
        axes[0].set_title('Image')

        axes[1].imshow(img)
        axes[1].imshow(all_masks[i][0].numpy(), cmap='Reds', alpha=0.5)   # ← all_masks
        axes[1].set_title('Ground Truth')

        axes[2].imshow(img)
        axes[2].imshow(all_preds[i][0].numpy(), cmap='Reds', alpha=0.5)   # ← all_preds
        axes[2].set_title('Prediction')

        for ax in axes:
            ax.axis('off')

        fig.tight_layout()
        plt.show()                                           # ← renders inline



## 7. Run Everything

In [ ]:
train_loader, test_loader = load_ultrasound_data(
    images_dir='../images',
    masks_dir='../masks',
    batch_size=4,
)
model = get_model()

In [ ]:
train_model(model, train_loader, epochs=30)

In [ ]:
plot_training_curves(history)

In [ ]:
print('Evaluating on test set:')
evaluate_and_visualize(model, test_loader)